In [53]:

from src.hypothesis_tests import chi_square_test, welch_ttest

import pandas as pd
import numpy as np
from scipy import stats

In [57]:
df = pd.read_csv(
    r"C:\Users\Soret\insurance-risk-analytics\data\cleaned_data.csv",
    low_memory=False
)

In [58]:
def prepare_data(df):
    df = df.copy()

    # KPI 1: Margin
    df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

    # KPI 2: Claim Frequency
    df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

    return df


df = prepare_data(df)

In [59]:
df[["TotalPremium", "TotalClaims", "Margin", "HasClaim"]].head()

,TotalPremium,TotalClaims,Margin,HasClaim
0,21.929825,0.0,21.929825,0
1,21.929825,0.0,21.929825,0
2,0.000000,0.0,0.000000,0
3,512.848070,0.0,512.848070,0
4,0.000000,0.0,0.000000,0


In [69]:
df[["Margin", "HasClaim"]].describe()

,Margin,HasClaim
count,990562.000000,990562.000000
mean,-3.304869,0.002800
std,2371.481005,0.052845
min,-392848.566930,0.000000
25%,0.000000,0.000000
50%,2.174825,0.000000
75%,21.929825,0.000000
max,65282.603421,1.000000


In [70]:
df["HasClaim"].value_counts()

HasClaim
0    987788
1      2774
Name: count, dtype: int64

In [62]:
df.columns

Index(['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth',
       'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language',
       'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province',
       'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode',
       'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders',
       'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors',
       'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser',
       'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff',
       'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet',
       'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm',
       'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section',
       'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium',
       'TotalClaims', 'Margin', 'HasClaim'],
      dtype='str')

In [63]:
df = df.dropna(subset=[
    "Province",
    "Gender",
    "PostalCode",
    "TotalClaims",
    "TotalPremium"
])

In [65]:
p1 = chi_square_test(df, "Province", "HasClaim")["p_value"]

In [66]:
print(p1)

9.268669871570131e-20


In [67]:
p2 = chi_square_test(df, "PostalCode", "HasClaim")["p_value"]

In [68]:
p2 = chi_square_test(df, "PostalCode", "HasClaim")["p_value"]
p2

4.099822466714019e-30

In [74]:
df["Postal3"] = df["PostalCode"].astype(str).str[:3]

In [75]:
postal_groups = df.groupby("Postal3")["Margin"].mean()
top2 = postal_groups.sort_values().index[:2]

group_a, group_b = top2[0], top2[1]

In [64]:
group_A_province = df["Province"].value_counts().idxmax()

provinces = [p for p in df["Province"].unique() if p != group_A_province]

group_B_province = min(
    provinces,
    key=lambda x: abs(len(df[df["Province"] == x]) - len(df[df["Province"] == group_A_province]))
)

In [72]:
p_gender = chi_square_test(df, "Gender", "HasClaim")["p_value"]

In [73]:
p_gender 

0.026570248768437145

In [6]:
group_A_zip = df["PostalCode"].value_counts().idxmax()

zips = [z for z in df["PostalCode"].unique() if z != group_A_zip]

group_B_zip = min(
    zips,
    key=lambda x: abs(len(df[df["PostalCode"] == x]) - len(df[df["PostalCode"] == group_A_zip]))
)

In [7]:
group_A_gender = df["Gender"].value_counts().idxmax()
group_B_gender = [g for g in df["Gender"].unique() if g != group_A_gender][0]

In [8]:
p_province = chi_square_test(df, "Province", "has_claim")

In [9]:
p_zip_freq = chi_square_test(df, "PostalCode", "has_claim")

In [10]:
p_zip_margin = t_test(df, "PostalCode", "margin", group_A_zip, group_B_zip)

In [11]:
def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [13]:
p_gender = chi_square_test(df, "Gender", "has_claim")
p_gender

np.float64(0.026570248768437145)

In [14]:
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_province, decision(p_province)],
    ["Zip vs Risk", "Claim Frequency", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "Margin", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Claim Frequency", p_gender, decision(p_gender)]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0


In [15]:
df["has_claim"] = (df["TotalClaims"] > 0).astype(int)

df["claim_frequency"] = df["has_claim"]

df["claim_severity"] = df["TotalClaims"]
df.loc[df["TotalClaims"] == 0, "claim_severity"] = None

df["margin"] = df["TotalPremium"] - df["TotalClaims"]

In [16]:
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

In [17]:
def chi_square_pvalue(df, feature, target):
    table = pd.crosstab(df[feature], df[target])
    chi2, p, dof, exp = chi2_contingency(table)
    return p


def t_test_pvalue(df, group_col, value_col, group_a, group_b):
    a = df[df[group_col] == group_a][value_col].dropna()
    b = df[df[group_col] == group_b][value_col].dropna()
    stat, p = ttest_ind(a, b, equal_var=False)
    return p

In [18]:
p_province = chi_square_pvalue(df, "Province", "has_claim")

In [19]:
p_zip_freq = chi_square_pvalue(df, "PostalCode", "has_claim")

In [20]:
group_A_zip = df["PostalCode"].value_counts().idxmax()
group_B_zip = df["PostalCode"].value_counts().index[1]

In [21]:
p_zip_margin = t_test_pvalue(df, "PostalCode", "margin", group_A_zip, group_B_zip)

In [22]:
p_gender = chi_square_pvalue(df, "Gender", "has_claim")

In [23]:
def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [24]:
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_province, decision(p_province)],
    ["Zip vs Risk", "Claim Frequency", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "Margin", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Claim Frequency", p_gender, decision(p_gender)]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0


In [87]:
df["Postal3"] = df["PostalCode"].astype(str).str[:3]

group_a = df[df["Postal3"] == "000"]   # baseline (choose most frequent group)
group_b = df[df["Postal3"] != "000"]

In [88]:
df["Postal3"]

0          145
1          145
2          145
3          145
4          145
          ... 
1000093    749
1000094    749
1000095    749
1000096    749
1000097    749
Name: Postal3, Length: 990562, dtype: str

In [89]:
group_a = df[df["Postal3"] == top2[0]]["Margin"]
group_b = df[df["Postal3"] == top2[1]]["Margin"]

In [90]:
group_a

854044       21.222411
854045       92.105263
854046      181.118509
873869      505.198596
873871        1.207456
877058        1.168506
906282       21.929825
951500        2.337012
951501       92.105263
951502        3.505518
951503        3.505518
964471        1.207456
964472        3.622368
964473   -38984.782343
981627        2.414912
981628        3.622368
981630      175.275976
981633        1.168506
Name: Margin, dtype: float64

In [83]:
def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [84]:
def prepare_data(df):
    df = df.copy()

    df["Margin"] = df["TotalPremium"] - df["TotalClaims"]
    df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
    df["Postal3"] = df["PostalCode"].astype(str).str[:3]

    return df


df = prepare_data(df)

In [91]:
group_a = df[df["Gender"].str.upper() == "FEMALE"]
group_b = df[df["Gender"].str.upper() == "MALE"]

In [92]:
group_a

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,Margin,HasClaim,Postal3
715815,120443,9980,2015-02-01 00:00:00,False,,Individual,Mrs,English,First National Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Commercial Cover: Monthly,Commercial,IFRS Constant,0.000000,0.0,0.000000,0,102
715817,120443,9980,2014-12-01 00:00:00,False,,Individual,Mrs,English,First National Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Commercial Cover: Monthly,Commercial,IFRS Constant,0.000000,0.0,0.000000,0,102
715819,120443,9980,2015-03-01 00:00:00,False,,Individual,Mrs,English,First National Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Commercial Cover: Monthly,Commercial,IFRS Constant,0.000000,0.0,0.000000,0,102
715821,120443,9980,2015-05-01 00:00:00,False,,Individual,Mrs,English,First National Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Commercial Cover: Monthly,Commercial,IFRS Constant,0.000000,0.0,0.000000,0,102
715823,120443,9980,2015-07-01 00:00:00,False,,Individual,Mrs,English,First National Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Commercial Cover: Monthly,Commercial,IFRS Constant,0.000000,0.0,0.000000,0,102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999950,52673,361,2015-02-01 00:00:00,False,ZA,Private company,Ms,English,Standard Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,479.622895,0.0,479.622895,0,740
999951,52674,361,2014-08-01 00:00:00,False,ZA,Private company,Ms,English,Standard Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,479.622895,0.0,479.622895,0,740
999952,52674,361,2014-10-01 00:00:00,False,ZA,Private company,Ms,English,Standard Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,479.622895,0.0,479.622895,0,740
999953,52674,361,2014-12-01 00:00:00,False,ZA,Private company,Ms,English,Standard Bank,Current account,...,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,479.622895,0.0,479.622895,0,740


In [93]:
df = df.dropna(subset=["Province", "Gender", "PostalCode"])

In [94]:
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

In [97]:
import sys
sys.path.append(".")

In [98]:
from src.hypothesis_tests import (
    add_margin,
    add_claim_frequency,
    chi_square_test,
    welch_ttest,
    results_table
)

In [99]:
df = add_margin(df)
df = add_claim_frequency(df)

In [100]:
p1 = chi_square_test(df, "Province", "has_claim")
p1

{'test': 'Chi-square',
 'group_column': 'Province',
 'target_column': 'has_claim',
 'chi2_statistic': 108.11909907234548,
 'degrees_of_freedom': 8,
 'p_value': 9.268669871570131e-20,
 'decision': 'Reject H0'}

In [101]:
p2 = chi_square_test(df, "Gender", "has_claim")
p2

{'test': 'Chi-square',
 'group_column': 'Gender',
 'target_column': 'has_claim',
 'chi2_statistic': 7.255926312995721,
 'degrees_of_freedom': 2,
 'p_value': 0.026570248768437145,
 'decision': 'Reject H0'}

In [102]:
df["Postal3"] = df["PostalCode"].astype(str).str[:3]

top2 = df["Postal3"].value_counts().index[:2]

p3 = welch_ttest(
    df,
    "Postal3",
    "margin",
    top2[0],
    top2[1]
)

p3

{'test': 'Welch t-test',
 'group_column': 'Postal3',
 'value_column': 'margin',
 'group_a': '200',
 'group_b': '122',
 't_statistic': 1.196339332321002,
 'p_value': 0.23156808939628531,
 'decision': 'Fail to Reject H0'}

In [103]:
results = results_table([
    p1,
    p2,
    p3
])

results

,test,group_column,target_column,chi2_statistic,degrees_of_freedom,p_value,decision,value_column,group_a,group_b,t_statistic
0,Chi-square,Province,has_claim,108.119099,8.0,9.268670e-20,Reject H0,NaN,NaN,NaN,NaN
1,Chi-square,Gender,has_claim,7.255926,2.0,2.657025e-02,Reject H0,NaN,NaN,NaN,NaN
2,Welch t-test,Postal3,NaN,NaN,NaN,2.315681e-01,Fail to Reject H0,margin,200,122,1.196339


In [104]:
results[[
    "test",
    "group_column",
    "p_value",
    "decision"
]]

,test,group_column,p_value,decision
0,Chi-square,Province,9.268670e-20,Reject H0
1,Chi-square,Gender,2.657025e-02,Reject H0
2,Welch t-test,Postal3,2.315681e-01,Fail to Reject H0


In [105]:
def prepare_kpis(df):
    df = df.copy()

    # Numerical KPI
    df["margin"] = df["TotalPremium"] - df["TotalClaims"]

    # Categorical KPI
    df["has_claim"] = (df["TotalClaims"] > 0).astype(int)

    # Postal grouping (fixes missing ZipCode issue)
    df["postal3"] = df["PostalCode"].astype(str).str[:3]

    return df


df = prepare_kpis(df)

In [106]:
def chi_square_test(df, group_col, target_col):
    table = pd.crosstab(df[group_col], df[target_col])
    chi2, p, _, _ = stats.chi2_contingency(table)

    return {
        "test": "chi-square",
        "feature": group_col,
        "p-value": p,
        "decision": "Reject H0" if p < 0.05 else "Fail to Reject H0"
    }

In [107]:
def t_test(df, group_col, value_col, group_a, group_b):
    a = df[df[group_col] == group_a][value_col].dropna()
    b = df[df[group_col] == group_b][value_col].dropna()

    t_stat, p = stats.ttest_ind(a, b, equal_var=False)

    return {
        "test": "t-test",
        "feature": group_col,
        "p-value": p,
        "decision": "Reject H0" if p < 0.05 else "Fail to Reject H0"
    }

In [108]:
r1 = chi_square_test(df, "Province", "has_claim")
r1

{'test': 'chi-square',
 'feature': 'Province',
 'p-value': np.float64(9.268669871570131e-20),
 'decision': 'Reject H0'}

In [109]:
r2 = chi_square_test(df, "Gender", "has_claim")
r2

{'test': 'chi-square',
 'feature': 'Gender',
 'p-value': np.float64(0.026570248768437145),
 'decision': 'Reject H0'}

In [110]:
top2 = df["postal3"].value_counts().index[:2]
sub = df[df["postal3"].isin(top2)]

r3 = chi_square_test(sub, "postal3", "has_claim")
r3

{'test': 'chi-square',
 'feature': 'postal3',
 'p-value': np.float64(0.028126173191600747),
 'decision': 'Reject H0'}

In [111]:
g1, g2 = top2[0], top2[1]

r4 = t_test(df, "postal3", "margin", g1, g2)
r4

{'test': 't-test',
 'feature': 'postal3',
 'p-value': np.float64(0.23156808939628531),
 'decision': 'Fail to Reject H0'}

In [112]:
g1, g2 = top2[0], top2[1]

r4 = t_test(df, "postal3", "margin", g1, g2)
r4

{'test': 't-test',
 'feature': 'postal3',
 'p-value': np.float64(0.23156808939628531),
 'decision': 'Fail to Reject H0'}

In [113]:
results = pd.DataFrame([r1, r2, r3, r4])
results

,test,feature,p-value,decision
0,chi-square,Province,9.268670e-20,Reject H0
1,chi-square,Gender,2.657025e-02,Reject H0
2,chi-square,postal3,2.812617e-02,Reject H0
3,t-test,postal3,2.315681e-01,Fail to Reject H0


In [114]:
def decision_rule(p_value, alpha=0.05):

    if p_value < alpha:
        return "Reject H0"

    else:
        return "Fail to Reject H0"

In [115]:
p_value = 0.023

decision = decision_rule(p_value)

print(decision)

Reject H0


In [116]:
results["Decision"] = results["p-value"].apply(decision_rule)

In [117]:
if p_value < 0.05:
    print("Reject H₀: statistically significant difference detected.")
else:
    print("Fail to Reject H₀: insufficient evidence of a significant difference.")

Reject H₀: statistically significant difference detected.


In [85]:
df = prepare_data(df)

In [28]:
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

In [30]:
results = [
    ("Province", p_province),
    ("Zip_Frequency", p_zip_freq),
    ("Zip_Margin", p_zip_margin),
    ("Gender", p_gender)
]

final_output = []

for h, p in results:
    final_output.append({
        "Hypothesis": h,
        "p-value": p,
        "Decision": decision(p),
        "Business Insight": business_interpretation(h, p)
    })

pd.DataFrame(final_output)

,Hypothesis,p-value,Decision,Business Insight
0,Province,5.925511e-19,Reject H0,We reject H₀ for provinces (p < 0.000). Geogra...
1,Zip_Frequency,3.152172e-30,Reject H0,We reject H₀ for zip codes (p < 0.000). Claim ...
2,Zip_Margin,6.630316e-01,Fail to reject H0,NaN
3,Gender,2.657025e-02,Reject H0,We reject H₀ for gender (p < 0.027). Claim fre...


In [35]:
results = [
    ["Province vs Risk", "Chi-square", p_province, decision(p_province)],
    ["Zip vs Risk", "Chi-square", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "t-test", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Chi-square", p_gender, decision(p_gender)]
]

In [78]:
results 

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0


In [79]:
import pandas as pd

results = pd.DataFrame(results, columns=[
    "Hypothesis", "Test", "p-value", "Decision"
])

In [80]:
results

,Hypothesis,Test,p-value,Decision
0,Province vs Risk,NaN,5.925511e-19,Reject H0
1,Zip vs Risk,NaN,3.152172e-30,Reject H0
2,Zip vs Margin,NaN,2.444624e-01,Fail to reject H0
3,Gender vs Risk,NaN,2.657025e-02,Reject H0


In [121]:
def business_interpretation(row):

    # use the correct column name
    if row["Decision"] != "Reject H0":
        return "No statistically significant difference detected."

    p = row["p-value"]

    feature = row["feature"]

    # Province
    if feature == "Province":

        return (
            f"We reject H₀ for provinces "
            f"(p = {p:.4f}). "
            "Claim frequency differs significantly across provinces, "
            "suggesting geographic risk factors materially affect claims. "
            "Regional premium adjustments may improve pricing accuracy."
        )

    # Postal
    elif feature in ["postal3", "Postal3"]:

        return (
            f"We reject H₀ for postal regions "
            f"(p = {p:.4f}). "
            "Certain postal regions exhibit significantly different risk "
            "and profitability patterns, indicating localized underwriting risk."
        )

    # Gender
    elif feature == "Gender":

        return (
            f"We reject H₀ for gender differences "
            f"(p = {p:.4f}). "
            "Claim frequency differs significantly between genders, "
            "suggesting variation in underlying insurance risk exposure."
        )

    else:
        return "Statistically significant difference detected."

In [120]:
print(results.columns)

Index(['test', 'feature', 'p-value', 'decision', 'Decision'], dtype='str')


In [122]:
results["Business Interpretation"] = results.apply(
    business_interpretation,
    axis=1
)

In [123]:
results

,test,feature,p-value,decision,Decision,Business Interpretation
0,chi-square,Province,9.268670e-20,Reject H0,Reject H0,We reject H₀ for provinces (p = 0.0000). Claim...
1,chi-square,Gender,2.657025e-02,Reject H0,Reject H0,We reject H₀ for gender differences (p = 0.026...
2,chi-square,postal3,2.812617e-02,Reject H0,Reject H0,We reject H₀ for postal regions (p = 0.0281). ...
3,t-test,postal3,2.315681e-01,Fail to Reject H0,Fail to Reject H0,No statistically significant difference detected.


In [127]:
results = results.drop(columns=["decision"])

In [129]:
results["Business Interpretation"] = results.apply(
    business_interpretation,
    axis=1
)

In [131]:
print(results.columns)

Index(['test', 'feature', 'p-value', 'Decision', 'Business Interpretation'], dtype='str')


In [132]:
results.columns = [
    "Test",
    "Feature",
    "p-value",
    "Decision",
    "Business Interpretation"
]

In [133]:
results

,Test,Feature,p-value,Decision,Business Interpretation
0,chi-square,Province,9.268670e-20,Reject H0,We reject H₀ for provinces (p = 0.0000). Claim...
1,chi-square,Gender,2.657025e-02,Reject H0,We reject H₀ for gender differences (p = 0.026...
2,chi-square,postal3,2.812617e-02,Reject H0,We reject H₀ for postal regions (p = 0.0281). ...
3,t-test,postal3,2.315681e-01,Fail to Reject H0,No statistically significant difference detected.


In [48]:
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_province, decision(p_province)],
    ["Zip vs Risk", "Claim Frequency", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "Margin", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Claim Frequency", p_gender, decision(p_gender)]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0
